# Analise Exploratoria de Dados (EDA) - F1 Pit Stops

Investigacao detalhada do dataset para prever se um piloto de Formula 1 fara uma parada nos boxes (pit stop) na proxima volta.

In [31]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Adiciona o diretorio raiz do projeto ao path para permitir imports do pacote local 'src'
root_dir = Path.cwd().parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src import config, data_loader, features

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Carregamento dos Dados

In [32]:
train = data_loader.load_train_data()
test = data_loader.load_test_data()
print(f"Treino: {train.shape}")
print(f"Teste: {test.shape}")

Treino: (439140, 16)
Teste: (188165, 15)


## Inspecao Inicial e Qualidade dos Dados

Visualizacao das primeiras linhas, verificacao dos tipos das variaveis e checagem de valores nulos no dataset de treino.

In [33]:
# 1. Visualizar as primeiras linhas do dataset
train.head()

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0


In [34]:
# 2. Verificacao dos tipos de dados de cada coluna
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 439140 entries, 0 to 439139
Data columns (total 16 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   id                      439140 non-null  int64  
 1   Driver                  439140 non-null  object 
 2   Compound                439140 non-null  object 
 3   Race                    439140 non-null  object 
 4   Year                    439140 non-null  int64  
 5   PitStop                 439140 non-null  int64  
 6   LapNumber               439140 non-null  int64  
 7   Stint                   439140 non-null  int64  
 8   TyreLife                439140 non-null  float64
 9   Position                439140 non-null  int64  
 10  LapTime (s)             439140 non-null  float64
 11  LapTime_Delta           439140 non-null  float64
 12  Cumulative_Degradation  439140 non-null  float64
 13  RaceProgress            439140 non-null  float64
 14  Position_Change     

In [35]:
# 3. Checagem explicita de valores nulos (missing values)
print("Valores ausentes por coluna:")
print(train.isnull().sum())

Valores ausentes por coluna:
id                        0
Driver                    0
Compound                  0
Race                      0
Year                      0
PitStop                   0
LapNumber                 0
Stint                     0
TyreLife                  0
Position                  0
LapTime (s)               0
LapTime_Delta             0
Cumulative_Degradation    0
RaceProgress              0
Position_Change           0
PitNextLap                0
dtype: int64


### Conclusao da Inspecao Inicial:
- **Sem Dados Ausentes:** Nao ha valores nulos em nenhuma das colunas do dataset (todas possuem 439.140 registros validos).
- **Tipos de Dados:** 
  - Colunas de texto (`Driver`, `Compound`, `Race`) sao carregadas como tipo `object` e foram mapeadas para `category` para melhor performance.
  - Variáveis temporais e contínuas estão devidamente representadas como `int64` ou `float64`.

## 1. Analise de Distribuicao e Target Balance

Analisando a variavel alvo `PitNextLap` para identificar se ha desbalanceamento acentuado.

In [36]:
target_counts = train[config.TARGET_COL].value_counts()
target_pct = train[config.TARGET_COL].value_counts(normalize=True) * 100

print("Contagem do Target:")
print(target_counts)
print("\nPorcentagem do Target:")
print(target_pct)

Contagem do Target:
PitNextLap
0.0    351759
1.0     87381
Name: count, dtype: int64

Porcentagem do Target:
PitNextLap
0.0    80.10179
1.0    19.89821
Name: proportion, dtype: float64


### Conclusao sobre o Target:
- O target `PitNextLap` e desbalanceado em uma proporcao de aproximadamente **80% (Voltas sem parada)** contra **20% (Voltas que antecedem parada)**.
- Uma proporcao de 4:1 e considerada moderadamente desbalanceada e perfeitamente tratavel com modelos baseados em gradiente de arvore (LightGBM/XGBoost) usando `StratifiedKFold` para divisao justa das dobras.

## 2. Analise do Desgaste de Pneus (TyreLife & Compounds)

Investigando a idade do composto de pneu e como ela varia dependendo do tipo de composto (`HARD`, `MEDIUM`, `SOFT`, `INTERMEDIATE`, `WET`).

In [37]:
print("TyreLife (idade do pneu em voltas) por Composto Geral:")
print(train.groupby("Compound")["TyreLife"].agg(["mean", "max", "count"]))

print("\nTyreLife nos momentos em que ocorrem PitStops na volta atual:")
print(
    train[train["PitStop"] == 1]
    .groupby("Compound")["TyreLife"]
    .agg(["mean", "max", "count"])
)

TyreLife (idade do pneu em voltas) por Composto Geral:
                   mean   max   count
Compound                             
HARD          17.848441  77.0  170518
INTERMEDIATE  13.228225  71.0   17382
MEDIUM        11.830294  76.0  211141
SOFT          11.180880  66.0   38744
WET            9.579336  31.0    1355

TyreLife nos momentos em que ocorrem PitStops na volta atual:
                   mean   max  count
Compound                            
HARD          11.756385  67.0  32030
INTERMEDIATE  10.352547  42.0   2238
MEDIUM         9.981328  50.0  20191
SOFT           8.793556  44.0   5028
WET            7.138889  30.0    288


### Conclusao sobre Pneus:
- **Durabilidade Esperada:** O composto `HARD` e o mais duradouro, com media de 17.8 voltas e maximo de 77 voltas. Ja compostos macios `SOFT` tem media geral menor de 11.1 voltas.
- **Comportamento no Pit Stop:** Nos momentos reais de parada de boxes (`PitStop == 1`), a media de voltas cai significativamente por composto:
  - `SOFT`: Troca media em **8.7 voltas**.
  - `MEDIUM`: Troca media em **9.9 voltas**.
  - `HARD`: Troca media em **11.7 voltas**.
- **Insights preditivos:** Esta relacao confirma que features combinando `TyreLife` e `Compound` sao determinantes para guiar as decisões de Pit Stop.

## 3. Comportamento por Stint da Corrida

Investigando a probabilidade de parada de boxes em relacao ao numero do Stint atual (sequencia de paradas).

In [38]:
print("Probabilidade de PitNextLap por Stint:")
print(train.groupby("Stint")["PitNextLap"].mean())

Probabilidade de PitNextLap por Stint:
Stint
1    0.059818
2    0.391104
3    0.293105
4    0.171666
5    0.053025
6    0.019231
7    0.000000
8    0.020000
Name: PitNextLap, dtype: float64


### Conclusao sobre Stints:
- O primeiro stint (`Stint == 1`) tem baixissima probabilidade de parada (apenas 5.9%), ja que os carros largam de pneus novos.
- O segundo stint (`Stint == 2`) apresenta um salto tremendo na probabilidade geral.
- Isso nos diz que a feature `Stint` e uma excelente variavel para segmentacao e interacao no modelo.

## 4. Analise de Correlacao Linear com o Target

Calculando a correlacao de Pearson para todas as colunas numericas em relacao ao target preditivo.

In [39]:
num_cols = train.select_dtypes(include=[np.number]).columns
corrs = train[num_cols].corr()[config.TARGET_COL].sort_values(ascending=False)
print("Correlacao com o Target (PitNextLap):")
print(corrs)

Correlacao com o Target (PitNextLap):
PitNextLap                1.000000
TyreLife                  0.273510
LapNumber                 0.267057
Stint                     0.198193
RaceProgress              0.185477
Year                      0.125267
PitStop                   0.048567
Position_Change           0.046230
Position                  0.021348
id                       -0.000097
LapTime_Delta            -0.004946
LapTime (s)              -0.034096
Cumulative_Degradation   -0.167401
Name: PitNextLap, dtype: float64


### Conclusao sobre Correlacoes:
- **Correlacoes Positivas Fortes:** `TyreLife` (+0.273) e `LapNumber` (+0.267) sao as variaveis individuais com maior correlacao direta positiva com a probabilidade de parar nos boxes na proxima volta.
- **Correlacao Negativa Forte:** `Cumulative_Degradation` possui uma correlacao negativa expressiva de **-0.167**. Isso sugere que menores valores de degradacao acumulada representam desgaste severo ou perda de aderencia (grip), sendo um gatilho fortissimo para pits.
- **Variaveis irrelevantes:** `id` possui correlacao nula (~0.000), confirmando que deve ser descartada da modelagem para evitar ruidos de memorizacao.

## 5. Exploração Avançada (Etapas A, B, C e D)

### Etapa A: Desgaste e Degradacao Conjunta dos Compostos

Investigando `TyreLife` e `Cumulative_Degradation` especificamente na janela onde a decisao de box ocorre (`PitNextLap == 1.0`).

In [40]:
print(
    "Valores medios de TyreLife e Degradacao no momento de decisao de Pit Stop (PitNextLap == 1.0):"
)
print(
    train[train["PitNextLap"] == 1.0]
    .groupby("Compound")[["TyreLife", "Cumulative_Degradation"]]
    .mean()
)

Valores medios de TyreLife e Degradacao no momento de decisao de Pit Stop (PitNextLap == 1.0):
               TyreLife  Cumulative_Degradation
Compound                                       
HARD          21.458640              -48.052333
INTERMEDIATE  18.043068              -45.642738
MEDIUM        17.136562              -38.499838
SOFT          12.612727              -30.313527
WET           13.029412              -29.833794


**Conclusao da Etapa A:**
- Para os pneus `SOFT`, a decisao de parada ocorre com media de **12.6 voltas** e degradação acumulada de **-30.31**.
- Para pneus `HARD`, a decisao e empurrada para **21.4 voltas** de media e degradação acumulada de **-48.05**.
- Isso demonstra uma diferenca linear clara por composto. Indica que features baseadas no desgaste relativo (ex: `TyreLife` dividido pela vida util media do composto) serao preditores de peso na Fase 2.

### Etapa B: Relacao entre Stint e Progresso da Corrida

Mapeando a taxa de paradas nos boxes segmentando a interacao entre o numero do `Stint` e frações percentuais do `RaceProgress` (divididos em quintis).

In [41]:
train["Progress_Bucket"] = pd.qcut(
    train["RaceProgress"],
    q=5,
    labels=["0-20%", "20-40%", "40-60%", "60-80%", "80-100%"],
)
pivot_stint_prog = (
    train.groupby(["Stint", "Progress_Bucket"], observed=False)["PitNextLap"]
    .mean()
    .unstack()
)
print("Taxa de PitNextLap cruzando Stint e Progresso da Corrida:")
print(pivot_stint_prog)

Taxa de PitNextLap cruzando Stint e Progresso da Corrida:
Progress_Bucket     0-20%    20-40%    40-60%    60-80%   80-100%
Stint                                                            
1                0.056828  0.067154  0.053051  0.063955  0.060961
2                0.151082  0.306614  0.410123  0.461897  0.340825
3                0.041667  0.102156  0.227377  0.356105  0.278413
4                0.023904  0.016176  0.033107  0.188905  0.190206
5                0.000000  0.000000  0.000000  0.086553  0.047447
6                0.000000  0.000000  0.000000  0.023256  0.019146
7                     NaN  0.000000       NaN       NaN  0.000000
8                     NaN       NaN       NaN       NaN  0.020000


**Conclusao da Etapa B:**
- No **Stint 1**, a taxa de parada e relativamente baixa no inicio da corrida, mas cresce progressivamente conforme o progresso se aproxima da janela esperada.
- No **Stint 2**, a taxa de paradas chega a um pico de **46.18%** na faixa de 60-80% do progresso da corrida, mas cai repentinamente para **34.08%** na reta final da corrida (80-100%).
- Isso prova que existe uma dinamica tática na reta final de corridas (evitando paradas proximas ao encerramento da prova), o que requer criacao de features de interacao entre Stint e progresso restante.

### Etapa C: Dinamica de Posicao na Pista e Ultrapassagens

Investigando se a taxa de decisao de pit stop varia dependendo da posicao atual do piloto na pista (`Position`).

In [42]:
print("Taxa de PitNextLap agrupada por posicao na pista:")
print(train.groupby("Position")["PitNextLap"].mean())

Taxa de PitNextLap agrupada por posicao na pista:
Position
1     0.159305
2     0.189947
3     0.192670
4     0.178533
5     0.192803
6     0.190179
7     0.195140
8     0.207249
9     0.206504
10    0.195466
11    0.203787
12    0.200465
13    0.235131
14    0.229894
15    0.219870
16    0.213377
17    0.204465
18    0.188796
19    0.168912
20    0.154098
Name: PitNextLap, dtype: float64


**Conclusao da Etapa C:**
- Pilotos em posicoes da ponta (P1 a P3) apresentam uma taxa estavel de **~16% a ~19%** de paradas na volta seguinte.
- A taxa de paradas cresce no meio do pelotao (P13 a P15), atingindo o pico de **23.51%** na P13.
- Ja pilotos nas ultimas duas posicoes (P19 e P20) mostram taxas menores (**15.4%** na P20).
- Isso reflete a tatica real das corridas: o pelotao intermediario arrisca muito mais paradas antecipadas (*undercuts*) para ganhar posições, enquanto os lanternas tentam postergar suas paradas ao maximo na esperanca de ganhar voltas gratuitas sob Safety Car.

### Etapa D: Assinatura de Desgaste Especifico por GP / Pistas (Circuit Bias)

Investigando a idade media do pneu e a degradacao media acumulada no momento exato em que ocorrem pit stops (`PitStop == 1`) agrupados por circuito GP (`Race`).

In [43]:
race_stats = (
    train[train["PitStop"] == 1]
    .groupby("Race")[["TyreLife", "Cumulative_Degradation"]]
    .agg(["mean", "count"])
)
print("Estatisticas de Pit Stop agrupadas por Grande Premio:")
print(race_stats.sort_values(("TyreLife", "mean")))

Estatisticas de Pit Stop agrupadas por Grande Premio:
                            TyreLife       Cumulative_Degradation      
                                mean count                   mean count
Race                                                                   
British Grand Prix          7.208000  1250             -19.463418  1250
Las Vegas Grand Prix        7.469888  1345              -4.579902  1345
Belgian Grand Prix          7.942494  1652             -27.508067  1652
Saudi Arabian Grand Prix    7.971491  2736             -20.461021  2736
Bahrain Grand Prix          8.402964  2901             -20.436976  2901
Chinese Grand Prix          9.407601  1342             -21.133187  1342
Monaco Grand Prix           9.483578  1431             -27.834101  1431
Spanish Grand Prix          9.637299  3110             -35.527515  3110
Italian Grand Prix         10.052465  2211             -41.196288  2211
Azerbaijan Grand Prix      10.417211   918             -22.803111   918
Canadian G

**Conclusao da Etapa D:**
- Circuitos como o *British Grand Prix* (Silverstone) e *Las Vegas Grand Prix* apresentam desgaste extremamente severo, forcando trocas de pneus muito cedo (media de **7.2 e 7.4 voltas**).
- Circuitos como o *Mexico City Grand Prix* sao muito mais suaves, esticando os pneus ate uma media de **14.3 voltas** antes da troca.
- **Conclusao Principal:** O circuito (`Race`) contem um forte bias estrategico de engenharia de pista. Usar a media historica de desgaste da pista como feature preditiva dara um enorme diferencial de ganho de performance para o classificador.

## 6. Explorações Adicionais (Etapas E, F, G e H)

### Etapa E: Queda de Desempenho (Drop-Off em LapTime & LapTime_Delta)

Investigando o comportamento do tempo de volta (`LapTime`) e da variação de ritmo (`LapTime_Delta`) nas voltas que antecedem um pit stop.

In [44]:
print(
    "Metricas de desempenho nas voltas normais (PitNextLap == 0.0) vs voltas de entrada de box (PitNextLap == 1.0):"
)
print(train.groupby("PitNextLap")[["LapTime (s)", "LapTime_Delta"]].mean())

Metricas de desempenho nas voltas normais (PitNextLap == 0.0) vs voltas de entrada de box (PitNextLap == 1.0):
            LapTime (s)  LapTime_Delta
PitNextLap                            
0.0           91.284747      -3.661698
1.0           89.596092      -4.206182


**Conclusao da Etapa E:**
- Nas voltas que antecedem a parada (`PitNextLap == 1.0`), a media de `LapTime (s)` e **mais rapida** (89.59s) em comparacao com as voltas normais (91.28s), acompanhada de um delta mais negativo (-4.20s).
- **Insight de Corrida:** Isso revela o efeito tático do **'in-lap push'** (a volta de entrada no box e a volta mais forte do stint, onde o piloto espreme toda a performance restante da borracha sabendo que vai trocar na volta seguinte). Mapear esse ganho de ritmo repentino sera uma feature de extrema utilidade!

### Etapa F: Regra de Pit Stop Consecutivo (Current PitStop vs PitNextLap)

Analisando a probabilidade de parar nos boxes na proxima volta dependendo se o piloto ja realizou uma parada na volta atual.

In [45]:
print("Probabilidade de PitNextLap baseada na coluna PitStop da volta atual:")
print(train.groupby("PitStop")["PitNextLap"].agg(["mean", "count"]))

Probabilidade de PitNextLap baseada na coluna PitStop da volta atual:
             mean   count
PitStop                  
0        0.191285  379365
1        0.247829   59775


**Conclusao da Etapa F:**
- Ao contrario da nossa intuicao de corrida real (onde paradas consecutivas sao quase impossiveis), o dataset apresenta uma taxa de pit stop consecutivo de **24.78%** para voltas onde `PitStop == 1`, contra 19.12% em voltas sem pit stop.
- **Conclusao Crucial:** Esta e uma caracteristica unica deste dataset especifico da competicao! Tentar forçar previsoes `0.0` logo apos paradas de boxes baseado na intuicao real de pista **prejudicaria** severamente o score do modelo. O EDA nos salvou de introduzir um viés incorreto no modelo que prejudicaria o score ROC AUC!

### Etapa G: Estilo de Preservacao do Piloto (Driver Tyre Preservation Bias)

Investigando a idade media do pneu (`TyreLife`) no momento em que a parada ocorre filtrada individualmente pelos IDs de pilotos (`Driver`).

In [46]:
driver_stats = (
    train[train["PitStop"] == 1].groupby("Driver")["TyreLife"].agg(["mean", "count"])
)
print("Top 5 pilotos com maior durabilidade de pneu na parada (min 30 pitstops):")
print(
    driver_stats[driver_stats["count"] >= 30]
    .sort_values("mean", ascending=False)
    .head(5)
)
print("\nTop 5 pilotos com menor durabilidade de pneu na parada (min 30 pitstops):")
print(
    driver_stats[driver_stats["count"] >= 30]
    .sort_values("mean", ascending=True)
    .head(5)
)

Top 5 pilotos com maior durabilidade de pneu na parada (min 30 pitstops):
             mean  count
Driver                  
OCO     22.000000     97
MAG     20.636364     55
LEC     20.173913    115
GAS     19.630435     92
ALB     19.545455     77

Top 5 pilotos com menor durabilidade de pneu na parada (min 30 pitstops):
            mean  count
Driver                 
D277    6.210526     38
D279    6.585366     41
D255    6.925000     40
D266    7.416667     36
D261    7.473684     38


**Conclusao da Etapa G:**
- Há uma variacao brutal no estilo de pilotagem! O piloto `OCO` consegue esticar a vida util media do pneu ate **22.0 voltas** antes da parada de box, enquanto o piloto `D277` faz paradas com media de apenas **6.2 voltas**.
- **Insight de Feature:** Criar uma pontuacao de preservacao de pneus por piloto (`Driver`) ou utilizar um Target Encoding robusto sera um dos maiores diferenciais de generalizacao preditiva no projeto.

### Etapa H: Mudanças e Tendencias de Durabilidade por Temporada (Yearly Trends)

Investigando se a vida util dos pneus no momento da parada real (`PitStop == 1`) variou sistematicamente entre as temporadas de 2022 a 2025.

In [47]:
print("Durabilidade media do pneu no pit stop agrupada por Ano da Temporada:")
print(train[train["PitStop"] == 1].groupby("Year")["TyreLife"].agg(["mean", "count"]))

Durabilidade media do pneu no pit stop agrupada por Ano da Temporada:
           mean  count
Year                  
2022  10.146890  15481
2023   2.379228   1685
2024  11.407359  24433
2025  11.428257  18176


**Conclusao da Etapa H:**
- Nas temporadas de `2022`, `2024` e `2025`, a durabilidade media dos pneus mantem-se estavel entre **10.1 e 11.4 voltas**.
- No entanto, a temporada de **`2023` apresenta um comportamento completamente anomalo: uma durabilidade media de apenas 2.37 voltas!** Alem disso, o volume de pit stops registrado e extremamente baixo (apenas 1.685 em 2023, contra mais de 15.000 nos outros anos).
- **Impacto de Modelagem:** O ano de 2023 contem uma discrepancia severa de distribuicao. Tratar a temporada (`Year`) com features categoricas ou normalizar as estimativas de desgaste separadamente por ano sera vital para nao confundir o modelo com o ruido anomalo de 2023.